# 📗 지식 그래프: 온톨로지와 추출·적재

앞 시간에는 트리플을 **손으로** 쪼개고, 노드의 키를 **id 로** 정하고, 시그니처를 **설계**했습니다. 이번 시간에는 그 설계를 실제로 돌립니다. 시그니처를 <strong>온톨로지</strong> 한 곳에 모으고, **구조화 출력**으로 논문에서 트리플을 받고, 뽑은 트리플을 **Neo4j 에 적재**합니다.

적재할 때 두 가지가 더 붙습니다. 사전에 없는 이름을 <strong>`:Candidate`</strong> 로 격리하는 일, 그리고 이 사실이 어디서 왔는지를 <strong>근거 등급(evidence_level)</strong>으로 남기는 일입니다. 의료 그래프에서 뒤엣것은 안전 문제입니다.

## ⏪ 복습: 지난 시간까지

- **트리플**: 사실 하나 = (주어, 관계, 목적어). 지식의 최소 단위.
- **LPG 매핑**: 주어·목적어는 노드, 관계는 대문자 스네이크 엣지. `MERGE` 라서 여러 번 적재해도 중복이 생기지 않습니다.
- **노드의 키는 id**: 이름은 개체를 가리키는 키가 못 됩니다. `obesity` 는 질병이면서 증상이라 이름만으로는 어느 쪽인지 정할 수 없습니다. 사전에 없는 이름은 `:Candidate` 로 격리하기로 했습니다.
- **관계 시그니처**: 관계마다 (주어 타입, 목적어 타입, 판정 기준). **오늘 1-1 에서** 이걸 한 곳에 모아 추출·적재가 함께 따르게 만들고, **3-1 에서** 그 시그니처로 적재 전에 거릅니다.

> **이 시간의 수치는 여러분 화면과 다를 수 있습니다.** 오늘부터 모델이 문서를 읽고 트리플을 뽑는데, 같은 문서·같은 프롬프트에도 **돌릴 때마다 조금씩 다르게** 답합니다. 그래서 아래에 적힌 건수·이름·타입은 **교재가 한 번 돌려 옮긴 기록**이지 맞춰야 할 정답이 아닙니다.
>
> 숫자를 교재와 맞추려 하지 마세요. 볼 것은 **어떤 트리플이 왜 걸러졌는가**입니다. 관계 이름이 허용 목록 밖이어서 걸린 것인지, 주어·목적어 타입이 시그니처와 어긋나서 걸린 것인지, 이름이 사전에 없어 격리된 것인지. 그 **이유**가 오늘 배울 내용이고, 건수는 그 이유가 몇 번 나왔는지를 셌을 뿐입니다.

> **데이터 출처**
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 본문 발췌 (`pgx_*.jsonl`) | PubMed Central Open Access Subset (pmcid 를 각 문서에 적어 두었습니다) | CC BY |
> | 이름 -> id 사전 (`name2id.json`) | Hetionet v1.0 (https://het.io) + RxNav(NLM) 약물 동의어 | CC0 / 공개 |
> | 큐레이션 관계 (`hetionet_curated.jsonl`) | Hetionet v1.0 에서 CC0 출처만 골라낸 부분 | CC0 |
>
> 논문에서 뽑은 관계는 **그 논문이 그렇게 보고했다**는 뜻이고, Hetionet 관계는 **2016년에 정리된** 문헌 근거라는 뜻입니다. 둘 다 "효능이 입증됐다"는 말이 아닙니다. 이 구분을 트리플 속성 `evidence_level` 로 남기는 법을 오늘 배웁니다.

**오늘의 목표**

**1. 규격 정하기: 온톨로지와 트리플 서식**
- [ ] (1-1) 시그니처를 **온톨로지 한 곳(dict)** 에 모으고, 그 dict 로 프롬프트 블록을 만든다.
- [ ] (1-2) **구조화 출력**으로 트리플을 `Extraction` 객체로 받는다(근거 evidence 포함).

**2. 추출 프롬프트 설계**
- [ ] (2-1) **온톨로지를 주입한 추출 프롬프트**를 설계하고, 판정 기준 한 줄이 결과를 어떻게 바꾸는지 본다.
- [ ] (2-2) **CoT** 로 추출을 단계로 쪼개고, `reasoning` 을 읽어 어디서 어긋났는지 찾는다.

**3. 그래프에 쌓기**
- [ ] (3-1) 뽑은 트리플을 **id 로 MERGE** 하고, 사전에 없는 이름은 **`:Candidate`** 로 격리한다.
- [ ] (3-1) 문서가 여럿이면 **`chain.batch`** 로 모델 호출을 묶고, **`max_concurrency`** 로 한 번에 내보낼 요청 수를 제한한다.
- [ ] (3-2) 관계에 **`evidence_level`**(curated/reported)을 남겨 근거의 무게를 구분한다.
- [ ] (3-3) **타입 계층**(`:Gene:Enzyme`)으로 시그니처를 안 고치고 받아들일 개체를 넓힌다.

아래 준비 셀 네 개를 차례로 실행하세요. 모델과 Neo4j 를 모두 씁니다.

세 번째 셀이 그래프를 초기화합니다. **이 단원은 지식그래프를 적재한 단원과 같은 레이블을 씁니다.** 그 그래프를 남겨 두고 싶으면 `.env` 의 `NEO4J_URI` 를 다른 실습 전용 데이터베이스로 바꾸세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] Neo4j 연결: .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 를 읽어 연결합니다.
# (자기 로컬 Neo4j 를 쓰면 기본값 그대로면 됩니다. 반드시 "실습 전용" DB 에 연결하세요.
#  아래 실습이 그래프를 지우고 새로 만듭니다.)
from neo4j import GraphDatabase

# os.getenv 의 두 번째 인자가 기본값이다. .env 에 값이 없으면 이 값으로 접속한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 연결이 안 되면 여기서 바로 에러가 난다. 뒤 셀까지 가지 않는다


def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # 값은 query 에 붙이지 않고 params 로 따로 넘긴다. 따옴표가 든 근거 문장도 안전하다
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 그래프 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
# 주의: 이 단원은 지식그래프를 적재한 단원과 같은 레이블을 씁니다. 그 그래프를 남겨 두고 싶으면
# .env 의 NEO4J_URI 를 다른 실습 전용 데이터베이스로 바꾸고 실행하세요.
DAY_LABELS = ["Compound", "Disease", "Gene", "Symptom", "PharmacologicClass",
              "Candidate", "Enzyme", "Transporter"]

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# 라벨 판정을 all 로 하는 이유: any 로 보면 한 라벨만 겹치는 남의 노드(:Gene:PatientRecord)가 통과합니다.
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=DAY_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 를 실습 전용 DB 로 바꾼 뒤 이 셀부터 다시 실행하세요.")

for label in DAY_LABELS:
    # DETACH 를 붙여야 그 노드에 붙은 관계까지 함께 지워진다(안 붙이면 관계가 있어 삭제가 거부된다)
    run_cypher(f"MATCH (n:{label}) DETACH DELETE n")
print("비운 레이블:", ", ".join(DAY_LABELS), "· 남은 노드 수:",
      run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"])


In [ ]:
# [제공 코드] 이름 -> id 사전 준비: 이 셀은 실행만 하세요.
# 사전을 어떻게 만드는지는 엔티티 정규화 단원에서 배웁니다. 여기서는 읽어 쓰기만 합니다.
import json
import re
from pathlib import Path

# 파일의 최상위 키는 여덟 개인데 이 노트북이 읽는 것은 셋이다
#   entries: 정규화한 이름 -> {id, label} / genes: 유전자 기호 -> id / ambiguous: 한 이름이 두 타입에 걸린 경우
NAME2ID = json.loads(Path("data/name2id.json").read_text(encoding="utf-8"))


# 표기 차이를 눌러 사전 조회용 키로 만든다. lookup_id 함수가 호출한다
def normalize_name(name):
    """이름 매칭용 정규화: 소문자, 앞뒤 공백 제거, 연속 공백 한 칸, 끝의 괄호 주석 제거."""
    s = re.sub(r"\s+", " ", name).strip().lower()
    return re.sub(r"\s*\([^)]*\)$", "", s).strip()


# 이름을 노드 id 로 바꿔 돌려준다. 트리플을 그래프에 적재할 때 쓴다
def lookup_id(name, node_type):
    """이름과 타입으로 Hetionet id 를 찾아 (id, 사유) 두 칸으로 돌려준다.

    사유는 셋 중 하나다. 못 붙은 이유가 둘로 갈리므로 id 만으로는 구별할 수 없다.
      "hit"        붙었다. 첫 칸이 그 id 다
      "miss"       사전에 그 이름이 없다(또는 타입이 다르다). 첫 칸은 None
      "ambiguous"  후보가 둘 이상이라 이름만으로는 못 고른다. 첫 칸은 None
    """
    if node_type == "Gene":
        # 유전자 기호는 대소문자가 곧 뜻이다. 소문자로 누르면 CAT·SET 같은 흔한 단어가 유전자로 잡힌다
        found = NAME2ID["genes"].get(name.strip())
        return (found, "hit") if found else (None, "miss")
    key = normalize_name(name)   # 유전자 말고는 표기 차이를 눌러 놓고 찾는다
    if key in NAME2ID["ambiguous"]:
        return None, "ambiguous"      # 한 이름이 서로 다른 타입 두 곳에 걸린 경우
    entry = NAME2ID["entries"].get(key)
    # 타입까지 맞아야 같은 개체다. obesity 는 Disease 이면서 Symptom 이라 타입을 안 보면 엉뚱하게 붙는다
    if entry and entry["label"] == node_type:
        return entry["id"], "hit"
    return None, "miss"


print("사전 항목:", len(NAME2ID["entries"]),
      "/ 유전자 기호:", len(NAME2ID["genes"]),
      "/ 애매한 이름:", len(NAME2ID["ambiguous"]))

## 오늘 쓸 논문 발췌
아래 셀은 **실행만** 하면 됩니다. 지난 시간에 개체를 뽑은 바로 그 논문 발췌를 다시 씁니다. 같은 지문을 쓰는 이유는, 다음 단원이 **이 지문에서 뽑은 트리플의 품질**을 재기 때문입니다.

In [ ]:
# [제공 코드] 논문 발췌 읽기: 실행만 하세요
# jsonl 은 한 줄이 문서 하나다. doc_id 를 키로 담아 두면 뒤에서 문서를 골라 쓰기 쉽다
core = {}
for line in Path("data/pgx_core.jsonl").read_text(encoding="utf-8").splitlines():
    row = json.loads(line)
    core[row["doc_id"]] = row

demo = core['PMC13368705']   # 오늘 데모로 쓸 문서 한 편
print(demo["doc_id"], "|", demo["journal"], demo["year"], "|", demo["license"])

In [ ]:
# [제공 코드] (이어서)
print(demo["title"])
print("-" * 60)

In [ ]:
# [제공 코드] (이어서)
print(demo["text"][:400], "...")

---
# 1. 규격 정하기: 온톨로지와 트리플 서식

지난 시간에 설계한 시그니처를 **코드 한 곳**으로 모으고, 모델이 답할 **모양**을 못 박습니다. 둘 다 뽑기 전에 정해 두는 규격입니다.

- **1-1** 시그니처를 dict 한 곳에 모으고, 그 dict 로 프롬프트에 끼울 블록을 만듭니다.
- **1-2** 구조화 출력으로 트리플을 객체로 받고, 서식이 강제하는 것과 못 하는 것을 가릅니다.

## 1-1. 온톨로지: 하나의 계약

<strong>온톨로지(ontology: 이 도메인에 어떤 타입과 관계가 있는지 못 박은 공식 규격)</strong>는 그래프DB 단원에서 이미 배웠습니다. **오늘 새로운 것은 그 규격을 코드 한 곳의 `dict` 로 두어, 추출·적재·평가·쿼리가 모두 같은 것을 읽게 만드는 것 하나입니다.**

지식 그래프를 만드는 일은 **추출 · 적재 · 평가 · 쿼리** 여러 단계로 나뉩니다. 이 넷이 서로 다른 규칙을 쓰면 그래프가 어긋납니다. 그래서 **모두가 따르는 계약 한 장**이 필요합니다.

### 문법: 코드 한 곳의 단일 계약
지난 시간에 만든 **허용 타입 + 관계 시그니처**가 온톨로지의 알맹이입니다. 핵심은 이걸 <strong>코드 한 곳(dict)</strong>에 두는 것: 여기만 고치면 추출·적재·평가·쿼리가 전부 따라 바뀝니다. 이렇게 규칙이 적힌 단 한 곳을 <strong>진실의 원천(single source of truth)</strong>이라고 부릅니다.

<img src="images/ontology_schema.png" width="700">

In [ ]:
# 관계 규격을 모아 두는 한 곳. 관계를 더하거나 고칠 때는 여기만 손댄다
# 각 관계: (주어 타입, 목적어 타입, 판정 기준)
RELATION_SIGNATURES = {
    # 아래 둘은 주어·목적어 타입이 같다. 셋째 칸(판정 기준)만이 둘을 가른다
    "TREATS":           ("Compound", "Disease",
                         "약이 질병을 치료한다. "
                         "질병의 원인이나 진행 자체에 작용한다"),
    "PALLIATES":        ("Compound", "Disease",
                         "약이 질병의 증상을 완화한다. "
                         "질병 자체는 그대로 두고 증상만 덜어 준다"),
    # 대사·수송을 담을 관계는 따로 두지 않았다. 어느 관계에 적을지를 판정 기준에 적어 BINDS 로 모은다
    "BINDS":            ("Compound", "Gene",
                         "약이 그 유전자의 단백질에 결합한다. "
                         "그 유전자가 이 약의 대사·수송을 맡는다는 진술도 여기에 적는다. "
                         "그 대신 표적·효소·수송체는 가르지 않는다"),
    "UPREGULATES_CG":   ("Compound", "Gene", "약이 그 유전자의 발현을 증가시킨다"),
    "DOWNREGULATES_CG": ("Compound", "Gene", "약이 그 유전자의 발현을 감소시킨다"),
    "ASSOCIATES":       ("Disease", "Gene", "질병과 유전자 사이에 연관이 보고됐다"),
    "PRESENTS":         ("Disease", "Symptom", "질병이 그 증상으로 나타난다"),
    "INCLUDES":         ("PharmacologicClass", "Compound",
                         "약효 분류가 그 약물을 포함한다. "
                         "그 약이 어느 계열에 속한다는 진술을 여기에 적는다"),
}

# 노드 타입 5종. 지식그래프를 적재한 단원의 레이블과 글자까지 같아야 한다
NODE_TYPES = {"Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"}

In [ ]:
def build_ontology_block(sigs):
    """시그니처 dict 를 프롬프트에 끼울 텍스트 블록으로 바꾼다."""
    lines = ["[허용 관계]"]      # 첫 줄은 제목. 아래로 관계를 한 줄씩 쌓는다
    for rel, (subj, obj, crit) in sigs.items():   # 값이 세 칸 튜플이라 for 문에서 바로 풀어 받는다
        # 판정 기준을 # 뒤에 함께 적는다. 모델이 관계를 고를 때 읽는 설명이 된다
        lines.append(f"- {rel}: ({subj}) -> ({obj})  # {crit}")
    return "\n".join(lines)


print(build_ontology_block(RELATION_SIGNATURES))

> 표로 봤던 시그니처가 그대로 **프롬프트에 붙일 텍스트**가 됐습니다. `dict` 를 고치면 이 블록도, 그걸 받는 추출 프롬프트도 저절로 바뀝니다.

### 🖐️ 함께 따라하기: 프로젝트 온톨로지로 같은 블록 만들기

**단위 프로젝트에서 만들 그래프로 연습합니다.** 파이썬 라이브러리 문서 그래프의 온톨로지는 관계도 노드 타입도 의학 그래프와 전혀 다릅니다. 아래 제공 셀이 그 온톨로지를 담아 둡니다.

`build_ontology_block` 을 **한 글자도 고치지 말고** 그 온톨로지에 적용해 블록을 출력하세요.

**확인 기준**: `[허용 관계]` 아래에 관계 **다섯 줄**이 찍히고, 각 줄이 `- 관계이름: (주어 타입) -> (목적어 타입)  # 판정 기준` 꼴입니다. 함수는 그대로인데 나온 블록이 통째로 달라집니다. **온톨로지가 dict 한 곳에 모여 있으면 도메인을 갈아 끼우는 일이 dict 교체 하나로 끝납니다.**

In [ ]:
# [제공 코드] 단위 프로젝트(라이브러리 문서 그래프)의 온톨로지: 실행만 하세요
# 관계도 노드 타입도 의학 그래프와 겹치는 것이 없다. 같은 것은 세 칸의 모양뿐이다
PYLIB_SIGNATURES = {
    "DEMONSTRATES": ("Document", "ApiElement", "문서가 그 API 를 예제로 보여 준다"),
    "RAISES":       ("ApiElement", "ApiElement", "그 API 가 이 예외를 낸다"),
    "INCLUDES":     ("Document", "Change", "릴리스 문서가 그 변경을 담고 있다"),
    "AFFECTS":      ("Change", "ApiElement", "그 변경이 이 API 를 바꾼다"),
    "REFERENCES":   ("Change", "Issue", "그 변경이 이 이슈를 가리킨다"),
}
PYLIB_NODE_TYPES = {"Document", "ApiElement", "Change", "Issue"}
print("프로젝트 관계:", list(PYLIB_SIGNATURES))

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) build_ontology_block 에 PYLIB_SIGNATURES 를 넘겨 블록을 만든다
# 2) 그 블록을 출력한다

### ✅ 바로 확인 퀴즈

**1.** 온톨로지를 **코드 한 곳의 단일 계약**으로 두는 가장 큰 이점은?

<details><summary>정답 보기</summary>

**한 곳만 고치면** 추출·적재·평가·쿼리가 함께 바뀌어, 여러 단계의 규칙이 어긋나지 않습니다.

</details>

**2.** 방금 따라하기에서 `build_ontology_block` 을 한 글자도 안 고치고 **전혀 다른 도메인의 온톨로지**를 넣었더니 블록이 통째로 바뀌었습니다. 왜 그럴까요?

<details><summary>정답 보기</summary>

블록을 **손으로 적어 둔 문자열**이 아니라 `build_ontology_block` 이 **dict 를 읽어 만들기** 때문입니다. 규격이 적힌 곳이 하나뿐이라 dict 와 프롬프트가 어긋날 수 없고, 도메인을 갈아 끼우는 일도 dict 교체 하나로 끝납니다.

</details>

## 1-2. 트리플 서식

구조화 출력은 지난 단원에서 배웠습니다. 오늘 새로운 것은 둘입니다. 서식이 담는 것이 **사실 하나**(여섯 칸)라는 것, 그리고 **값이 정해진 칸은 온톨로지에서 그대로 좁힌다**는 것.

먼저 좁히지 않은 서식으로 한 번 뽑아, 무엇이 새는지 봅니다.

In [ ]:
from pydantic import BaseModel, Field

# 칸 이름과 자료형만 정한 서식. 값이 무엇이어야 하는지는 아직 아무 데도 없다
class LooseTriple(BaseModel):
    subject: str      = Field(description="주어. 논문에 적힌 표기 그대로")
    subject_type: str = Field(description="주어 타입")
    relation: str     = Field(description="관계")
    object: str       = Field(description="목적어. 논문에 적힌 표기 그대로")
    object_type: str  = Field(description="목적어 타입")
    evidence: str     = Field(description="근거가 된 원문 (200자 이내)")


class LooseExtraction(BaseModel):
    triples: list[LooseTriple] = Field(description="문서에서 뽑은 트리플 목록")

In [ ]:
# 온톨로지 없이 문장 하나만 던져 본다. 관계 이름이 어떻게 나오는지 보는 것이 목적이다
text = 'Of 49 individuals taking simvastatin, 12 had a decreased activity SLCO1B1 intermediate transporter/decreased function (IT) phenotype.'
print(text)
print("-" * 60)

In [ ]:
loose_extractor = make_model().with_structured_output(LooseExtraction)
# 지시는 한 줄뿐이고 서식도 값을 안 막았으니 관계 이름은 모델 마음이다
loose_result = loose_extractor.invoke(f"다음 문서에서 트리플을 뽑아 줘.\n{text}")

# 관계 이름과 두 타입을 눈여겨보라. 허용 관계를 안 알려 줬으니 우리 규격 밖일 것이다
for tp in loose_result.triples:
    print(f"({tp.subject}, {tp.relation}, {tp.object})  타입: {tp.subject_type}, {tp.object_type}")

> 관계 이름을 보세요. `associated_with` 같은 이름은 **우리 온톨로지에 없습니다.** 목적어도 개체 이름이 아니라 서술문 조각이 통째로 들어왔습니다. 서식이 정한 것이 **칸 이름과 자료형까지**여서, `relation: str` 은 "문자열이면 된다"까지만 말했기 때문입니다. (결과는 실행마다 조금씩 다릅니다.)

### 문법: 값이 정해진 칸은 온톨로지에서 좁힌다
`relation` 에 올 수 있는 값은 이미 `RELATION_SIGNATURES` 에 있고, 타입 칸에 올 값은 `NODE_TYPES` 에 있습니다. 그 목록을 `Literal` 로 옮기면 JSON Schema 의 **`enum`** 이 되어 모델에게 그대로 전달됩니다. 목록을 손으로 다시 적지 않고 **온톨로지에서 가져오는 것**이 핵심입니다. 관계를 하나 더하면 서식도 따라 넓어집니다.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field

# 온톨로지 dict 에서 값 집합을 그대로 가져온다. 관계를 더하면 서식도 따라 넓어진다
RelationName = Literal[tuple(RELATION_SIGNATURES)]
NodeType = Literal[tuple(sorted(NODE_TYPES))]   # 집합은 순서가 없어 sorted 로 고정한다


# 트리플 서식: 필드 이름이 곧 모델이 채울 칸이 된다(칸 이름을 바꾸면 모델 답의 칸도 바뀐다)
class Triple(BaseModel):
    # Field 의 description 은 사람용 주석이 아니라 모델에게 그대로 전달되는 지시다
    subject: str           = Field(description="주어. 논문에 적힌 표기 그대로")
    subject_type: NodeType = Field(description="주어 타입")
    relation: RelationName = Field(description="관계. 온톨로지의 허용 관계 중 하나")
    object: str            = Field(description="목적어. 논문에 적힌 표기 그대로")
    object_type: NodeType  = Field(description="목적어 타입")
    # 근거를 200자로 묶는다. 문단을 통째로 담으면 "이 문장이 이 트리플을 입증하는가"를 못 따진다
    evidence: str          = Field(description="근거가 된 원문 (200자 이내)")


# 문서 하나에서 사실이 여러 개 나오므로 트리플을 리스트로 받는다
# 모델에 넘길 서식은 이 바깥 클래스다(안의 Triple 은 리스트 원소의 틀)
class Extraction(BaseModel):
    triples: list[Triple] = Field(description="문서에서 뽑은 트리플 목록")

In [ ]:
# 모델에 넘어가는 것은 파이썬 클래스가 아니라 JSON Schema 다. relation 칸에 무엇이 실렸는지 본다
print(json.dumps(Triple.model_json_schema()["properties"]["relation"], ensure_ascii=False, indent=1))

> `enum` 에 허용 관계 여덟이 그대로 실렸습니다. 이제 목록 밖 이름은 **답에 나올 수가 없습니다.**
>
> 다만 서식이 보는 것은 **칸 하나하나의 값**까지입니다. `PALLIATES` 의 목적어가 `Disease` 여야 한다는 **칸 사이의 약속**은 못 봅니다. 그건 3-1 의 시그니처 검사가 맡습니다.
>
> 그리고 값을 좁혔으니 **맞는 관계가 없을 때 어떻게 할지를 프롬프트가 정해 줘야** 합니다. 목록만 주고 그 말을 안 하면 모델이 목록 안에서 아무거나 고릅니다. 2-1 의 규칙에 그 줄이 들어갑니다.

### 🖐️ 함께 따라하기: 첫 트리플의 칸을 하나씩 꺼내기

위에서 받은 `loose_result.triples[0]` 의 `subject`·`subject_type`·`relation`·`object`·`object_type` 을 값과 타입을 짝지어 **세 줄로** 출력해 보세요(`relation` 은 타입이 없으니 그 한 줄은 값만). 새로 모델을 호출하지 말고 이미 받은 `loose_result` 를 씁니다.

관계 이름이 허용 관계인지는 아무도 보장하지 않은 상태이고, **타입 칸**은 뒤에서 시그니처와 대조할 때 쓰입니다. 둘 다 눈으로 확인해 두세요.

**확인 기준**: 세 줄이 찍히고 값과 타입이 짝지어 나옵니다. 값 자체는 모델이 그때그때 다르게 뽑으므로 **여러분 화면과 여기 적힌 것이 다를 수 있습니다.** 볼 것은 `관계` 칸입니다. `RELATION_SIGNATURES` 에 없는 이름(예: `associated_with`)이 들어와 있으면, 그것이 바로 다음 절에서 막을 문제입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) first = loose_result.triples[0] 로 첫 트리플을 꺼낸다
# 2) 주어와 그 타입, 관계, 목적어와 그 타입을 세 줄로 출력한다

### ✅ 바로 확인 퀴즈

**1.** 프롬프트로 "JSON으로 답해 줘"라고 부탁하는 것과 `with_structured_output` 의 가장 큰 차이는?

<details><summary>정답 보기</summary>

부탁은 지켜질지 불확실하지만, 구조화 출력은 **정한 필드·타입의 틀을 강제**합니다.

</details>

**2.** `LooseTriple` 로 뽑았을 때는 규격 밖 관계 이름이 나왔는데, `Triple` 로 바꾸면 그 일이 생기지 않습니다. 그런데도 3-1 에 시그니처 검사가 따로 필요한 이유는?

<details><summary>정답 보기</summary>

`LooseTriple` 의 `relation: str` 은 "문자열이면 된다"까지만 말해서 아무 이름이나 통과했습니다. `Triple` 은 그 칸을 허용 목록으로 좁혔으니 목록 밖 이름은 이제 안 나옵니다. 다만 서식이 보는 것은 **칸 하나하나의 값**까지입니다. `PALLIATES` 의 목적어가 `Disease` 여야 한다는 **칸 사이의 약속**은 서식으로 적을 수 없습니다. 그래서 시그니처 검사가 따로 필요합니다.

</details>

---
# 2. 추출 프롬프트 설계

**1. 규격 정하기**에서 정해 둔 것도 모델이 읽지 않으면 소용이 없습니다. 여기서는 그 규격을 **프롬프트에 실어** 보내고, 생각 순서까지 적어 주면 무엇이 달라지는지 봅니다.

- **2-1** 온톨로지를 주입한 추출 프롬프트를 설계하고, 판정 기준 한 줄이 결과를 어떻게 바꾸는지 봅니다.
- **2-2** CoT 로 추출을 단계로 쪼개고, 도움이 되는지는 재 봐야 안다는 것까지 확인합니다.

## 2-1. 온톨로지 주입

구조화 출력은 답의 **모양**을 강제하지만, **무엇을 어떻게 뽑을지**는 여전히 프롬프트가 정합니다. 좋은 추출 프롬프트는 <strong>재료(온톨로지)</strong>와 <strong>조리법(규칙)</strong>을 한 장에 담습니다.

### 문법: 추출 프롬프트의 네 부분
- **역할**: "너는 의학 논문에서 트리플을 뽑는 도구야"
- **온톨로지 블록**: `build_ontology_block` 이 만든 허용 관계 목록을 그대로 끼웁니다.
- **규칙**: 세 줄입니다.
  - 허용 관계 중 하나만 쓰고 **방향**(주어 타입 -> 목적어 타입)을 지킨다.
  - **개체 이름만** 넣는다: 서술문을 통째로 넣지 말라는 뜻입니다. "SLCO1B1 활성 저하 표현형"이 아니라 `SLCO1B1`.
  - 근거(evidence)는 원문을 200자 이내로 인용한다.
- **텍스트**: 트리플을 뽑을 실제 논문 발췌.

온톨로지 블록을 **코드로 끼워 넣기** 때문에, dict 를 고치면 프롬프트도 저절로 최신 규격이 됩니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate


def build_extraction_prompt():
    """역할·온톨로지·규칙을 담은 추출 프롬프트 템플릿을 돌려준다(문서는 실행할 때 넣는다)."""
    # dict 를 고치면 이 블록이 따라 바뀌고, 템플릿도 저절로 최신 규격이 된다
    ontology = build_ontology_block(RELATION_SIGNATURES)
    # f-문자열이라 {ontology} 자리에 위 블록이 들어간다. {text} 는 템플릿 변수로 남는다
    system = ("너는 의학 논문에서 트리플을 뽑는 도구야.\n\n"
              f"{ontology}\n\n"
              "[규칙]\n"
              "- 관계는 허용 관계 중 하나만 쓰고, 방향(주어 타입 -> 목적어 타입)을 지킨다.\n"
              # 서식이 관계 이름을 허용 목록으로 좁혀 두었으니, 맞는 것이 없을 때 어떻게 할지를 규칙이 정한다.
              # 이 줄이 없으면 맞는 관계가 없을 때 모델이 목록 안에서 아무거나 고른다
              "- 맞는 관계가 없으면 그 사실은 아예 넣지 않는다. 억지로 고르지 않는다.\n"
              "- 서술문을 통째로 넣지 말고 개체 이름만 넣는다. 이름은 논문 표기 그대로 쓴다.\n"
              "- 근거(evidence)는 원문을 200자 이내로 그대로 인용한다.")
    # {text} 만 변수로 남긴다. 실행할 때 invoke({"text": ...}) 로 채운다
    return ChatPromptTemplate.from_messages([("system", system), ("human", "[문서]\n{text}")])


# 자리표시 문자열을 넣으면 프롬프트 뼈대만 보인다
# 템플릿이 실제로 무엇을 보내는지 본다. 문서 자리는 {text} 로 남아 있다
for message in build_extraction_prompt().format_messages(text="(문서 자리)"):
    print(f"[{type(message).__name__}]")
    print(message.content)

### 근거(evidence)를 왜 트리플마다 붙일까요?
모델은 문서에 없는 사실을 그럴듯하게 **지어내기도** 합니다(환각). 근거가 있으면 "이 원문이 정말 이 트리플을 **입증하는가**"로 한 건씩 따질 수 있습니다. 그래서 품질을 잴 때 기준은 "세상에서 사실인가"가 아니라 <strong>"그 근거 문장이 이 트리플을 입증하는가"</strong>입니다. 적재할 때도 근거를 관계 속성으로 함께 저장합니다.

의학 문서에서는 이 구분이 특히 중요합니다. 뒤에서 보겠지만, 논문이 "연구되었다"고만 쓴 문장을 추출기가 "치료한다"로 적어 오는 일이 실제로 일어납니다.

In [ ]:
doc = demo["text"]   # 데모 논문 발췌 전문(앞에서 400자만 봤던 그 문서)

# 이번에는 값까지 좁힌 Extraction 서식에, 온톨로지를 주입한 템플릿을 이어 붙인다
extractor = make_model().with_structured_output(Extraction)
extraction_chain = build_extraction_prompt() | extractor
doc_result = extraction_chain.invoke({"text": doc})
print("뽑은 트리플:", len(doc_result.triples), "건")

In [ ]:
# 대괄호 안이 타입이다. 시그니처의 (주어 타입 -> 목적어 타입)과 맞는지 여기서 눈으로 대조한다
for tp in doc_result.triples:
    print(f"({tp.subject}[{tp.subject_type}], {tp.relation}, {tp.object}[{tp.object_type}])")

> 온톨로지를 주입하자 관계 이름이 **허용 관계**로 바뀌었고, 목적어도 서술문이 아니라 개체 이름이 됐습니다. 뽑힌 트리플이 모두 시그니처의 방향(Compound -> Gene)에 맞습니다.
>
> **판정 기준 한 줄이 결과를 갈랐습니다.** 이 논문의 문장들은 대부분 "그 유전자가 이 약의 대사·수송을 맡는다"는 말인데, 우리 관계 8종에는 대사가 없습니다. `BINDS` 의 판정 기준에 "대사·수송 진술도 여기에 적는다"를 적어 두지 않았다면, 모델은 어디에 적을지 몰라 관계 이름을 지어내거나 그 사실을 통째로 버렸을 것입니다.
>
> **그리고 그 대가가 이 줄들에 그대로 보입니다.** 화면에 나온 유전자를 하나씩 보세요. `CYP2D6`·`CYP2C19`·`CYP3A4` 처럼 약을 **분해하는 효소**도 있고, `SLCO1B1` 처럼 약을 **실어 나르는 수송체**도 있고, `VKORC1` 처럼 약이 **실제로 노리는 표적**도 있습니다. 성격이 다른데 그래프에서는 전부 `BINDS` 라 구분이 없습니다. 지난 시간에 본 그 대가입니다. (어느 유전자가 뽑히는지는 실행마다 조금씩 다릅니다.)
>
> 그렇다고 검사가 필요 없다는 뜻은 아닙니다. **이 문서에서 규격 밖 트리플이 안 나왔을 뿐**입니다. 3-1 에서 다른 논문을 넣어 보면 시그니처를 어긴 트리플이 실제로 나옵니다.

### 🖐️ 함께 따라하기: 프로젝트 문서에서 뽑아 보기

**여기서 오늘 배운 것이 단위 프로젝트로 넘어갑니다.** 온톨로지가 바뀌면 서식도 프롬프트도 따라 바뀝니다. 아래 제공 셀이 프로젝트 온톨로지로 좁힌 서식(`PylibExtraction`)과 문서 두 편을 준비해 둡니다.

- `build_pylib_prompt()` 를 만드세요. 본문의 `build_extraction_prompt()` 와 **같은 모양**인데 온톨로지 블록만 `PYLIB_SIGNATURES` 로 바꿉니다(규칙 네 줄은 그대로).
- 그 템플릿과 `PylibExtraction` 서식을 이어 붙여 `pylib_chain` 을 만드세요.
- `pylib_docs[0]`(seaborn 매뉴얼)의 `text` 로 뽑아 `pylib_result` 에 담고, `(주어[타입], 관계, 목적어[타입])` 꼴로 출력하세요.

**확인 기준**: 관계가 전부 `DEMONSTRATES`·`RAISES` 중 하나입니다. 이 문서는 **사용 축**이라 변경 축 관계(`INCLUDES`·`AFFECTS`·`REFERENCES`)는 나올 자리가 없습니다. 서식이 관계 이름을 다섯으로 좁혀 두었으니 목록 밖 이름은 **나올 수가 없습니다.** 건수는 실행마다 다릅니다.

**주어를 눈여겨보세요.** 실행마다 `문서`·`rugplot`·`Add a rug along one of the axes` 처럼 제각각일 것입니다. 이 온톨로지에서 `Document` 노드의 유일키는 **문서 제목이 아니라 `doc_id`** 인데, 모델은 `doc_id` 를 알 방법이 없습니다. **문서 앵커는 모델이 정하는 값이 아니라 파이프라인이 채우는 값**입니다. 3-1 의 `source_doc_id` 와 같은 자리이고, 단위 프로젝트에서 실제로 그렇게 채우게 됩니다.

> 그래서 3-1 의 `ground_check` 를 이 결과에 그대로 걸면 **전부 걸러집니다.** 근거는 코드 예제 한 줄인데 주어는 문서 이름이라, 주어가 근거 안에 있을 수가 없기 때문입니다. **검사도 온톨로지에 맞춰 고쳐야 합니다.** 문서 앵커가 주어인 관계는 목적어만 근거와 대조하는 것이 맞습니다.

In [ ]:
# [제공 코드] 프로젝트 온톨로지로 좁힌 서식과 문서 두 편: 실행만 하세요
PylibRelation = Literal[tuple(PYLIB_SIGNATURES)]
PylibNodeType = Literal[tuple(sorted(PYLIB_NODE_TYPES))]


class PylibTriple(BaseModel):
    subject: str                = Field(description="주어. 문서에 적힌 표기 그대로")
    subject_type: PylibNodeType = Field(description="주어 타입")
    relation: PylibRelation     = Field(description="관계. 온톨로지의 허용 관계 중 하나")
    object: str                 = Field(description="목적어. 문서에 적힌 표기 그대로")
    object_type: PylibNodeType  = Field(description="목적어 타입")
    evidence: str               = Field(description="근거가 된 원문 (200자 이내)")


class PylibExtraction(BaseModel):
    triples: list[PylibTriple] = Field(description="문서에서 뽑은 트리플 목록")


pylib_docs = [json.loads(line) for line
              in Path("data/pylibs_docs.jsonl").read_text(encoding="utf-8").splitlines()]
for d in pylib_docs:
    print(d["library"], "|", d["doc_type"], "|", d["title"], f"({len(d['text'])}자)")

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) build_pylib_prompt() 를 정의한다. 본문 build_extraction_prompt 와 같은 모양이고
#    온톨로지 블록만 PYLIB_SIGNATURES 로 바꾼다(규칙 네 줄은 그대로)
# 2) 그 템플릿과 PylibExtraction 서식을 이어 붙여 pylib_chain 을 만든다
# 3) pylib_docs[0]['text'] 로 뽑아 pylib_result 에 담는다
# 4) (주어[타입], 관계, 목적어[타입]) 꼴로 출력한다

### ✅ 바로 확인 퀴즈

**1.** 트리플마다 <strong>evidence(근거 원문)</strong>를 남기는 가장 중요한 이유는?

<details><summary>정답 보기</summary>

근거 문장이 그 트리플을 **입증하는지**로 품질을 채점할 수 있기 때문입니다(환각을 걸러낼 잣대).

</details>

**2.** 어떤 트리플의 `evidence` 가 **원문에 없는 문장**이었습니다. 무엇을 의심해야 할까요?

<details><summary>정답 보기</summary>

모델이 **근거까지 지어낸** 경우입니다. 근거가 원문에 실제로 있는지 대조하는 검사가 없으면 환각을 걸러낼 수 없습니다.

</details>

---
## 2-2. CoT 로 추출을 단계로 쪼개기

규칙을 적어 두어도, 한 문단에 사실이 여러 개 얽혀 있으면 모델이 관계를 건너뛰거나 방향을 헷갈립니다. `ASSOCIATES` 는 Disease -> Gene 인데 주어·목적어를 뒤집어 적는 식입니다. 이때 쓰는 기법이 <strong>CoT(Chain-of-Thought, 생각의 사슬)</strong>입니다. 답을 곧장 쓰게 하지 않고 **중간 단계를 거쳐** 답하게 합니다.

### 문법: 생각할 칸을 서식 앞에 둔다
추출은 이렇게 네 단계로 쪼갭니다.

1. 문서에 나온 **개체를 타입과 함께 나열**한다
2. 개체 쌍 중 **관계 후보**를 고른다
3. 후보마다 **시그니처와 대조**한다
4. 통과한 것만 **최종 트리플**로 적는다

문제는 구조화 출력에 **생각할 자리가 없다**는 점입니다. 칸이 곧 답이니까요. 그래서 서식 맨 앞에 `reasoning` 칸을 하나 둡니다. 모델은 칸을 **차례로** 채우므로, 앞 칸에서 생각을 한 뒤 뒤 칸에 트리플을 적게 됩니다. **칸 순서가 곧 생각 순서**입니다.

In [ ]:
# 트리플 서식(Triple)은 그대로 쓴다. 바뀐 것은 그 앞에 생각할 칸이 하나 붙은 것뿐이다
class CoTExtraction(BaseModel):
    # 앞 칸부터 채워지므로, 여기에 적은 생각이 뒤 칸(triples)의 답에 반영된다
    reasoning: str        = Field(description="1) 개체 나열 2) 관계 후보 3) 시그니처 대조. "
                                              "단계마다 그렇게 판정한 이유를 한 줄씩 함께 적는다")
    triples: list[Triple] = Field(description="시그니처를 통과한 최종 트리플만")

In [ ]:
def build_cot_prompt():
    """생각 순서를 단계로 적어 준 추출 프롬프트 템플릿을 돌려준다."""
    # 앞 절 템플릿과 다른 곳은 [생각 순서] 하나뿐이다. [규칙] 네 줄은 그대로 둬야
    # 결과가 달라졌을 때 원인을 CoT 로 돌릴 수 있다
    ontology = build_ontology_block(RELATION_SIGNATURES)
    system = ("너는 의학 논문에서 트리플을 뽑는 도구야.\n\n"
              f"{ontology}\n\n"
              "[생각 순서]\n"
              "1) 문서에 나온 개체를 타입과 함께 나열한다.\n"
              "2) 개체 쌍 중 관계가 있어 보이는 후보를 고른다.\n"
              "3) 후보마다 허용 관계의 (주어 타입, 목적어 타입)과 맞는지 대조한다.\n"
              "4) 통과한 것만 트리플로 적는다.\n\n"
              "[규칙]\n"
              "- 관계는 허용 관계 중 하나만 쓰고, 방향(주어 타입 -> 목적어 타입)을 지킨다.\n"
              "- 맞는 관계가 없으면 그 사실은 아예 넣지 않는다. 억지로 고르지 않는다.\n"
              "- 서술문을 통째로 넣지 말고 개체 이름만 넣는다. 이름은 논문 표기 그대로 쓴다.\n"
              "- 근거(evidence)는 원문을 200자 이내로 그대로 인용한다.\n"
              "- 1)~3) 은 reasoning 칸에 짧게 적고, triples 칸에는 4) 의 결과만 담는다.")
    return ChatPromptTemplate.from_messages([("system", system), ("human", "[문서]\n{text}")])

In [ ]:
# 서식이 바뀌었으니 추출기도 새로 만든다(같은 모델, 다른 틀)
cot_extractor = make_model().with_structured_output(CoTExtraction)
cot_chain = build_cot_prompt() | cot_extractor
# 호출은 한 번이다. 네 번 나눠 부르는 게 아니라 한 답 안에서 순서를 밟게 한 것뿐이다
cot_result = cot_chain.invoke({"text": doc})

print(cot_result.reasoning)
print("-" * 60)

In [ ]:
print("뽑은 트리플:", len(cot_result.triples), "건")

In [ ]:
for tp in cot_result.triples:
    print(f"({tp.subject}[{tp.subject_type}], {tp.relation}, {tp.object}[{tp.object_type}])")

> 호출은 **한 번**입니다. 네 번 나눠 부른 게 아니라, 한 번의 답 안에서 순서를 밟게 했을 뿐입니다.
>
> 앞 절 결과와 나란히 놓고 보세요. 이 문서에서는 **거의 그대로**입니다. `reasoning` 을 읽어 보면 3단계에서 이미 "약물-유전자 대사·수송 관계는 BINDS" 로 정리해 버렸습니다. 판정 기준이 또렷한 문서에서는 CoT 가 더 보탤 것이 없습니다.
>
> **그러면 CoT 는 쓸모없을까요?** 그렇지 않습니다. 값어치는 결과가 아니라 **`reasoning` 그 자체**에 있습니다. 모델이 어느 단계에서 무엇을 근거로 정했는지가 글자로 남아, 결과가 틀렸을 때 어디를 고쳐야 할지 짚을 수 있습니다. 다음 따라하기에서 그것이 실제로 필요한 장면을 봅니다.
>
> CoT 는 공짜가 아닙니다. `reasoning` 만큼 출력 토큰이 늘어 응답이 느려지고 비용도 올라갑니다. **아무 데나 붙이는 기법이 아닙니다.**

### 🖐️ 함께 따라하기: 모델을 바꿔 같은 프롬프트를 다시 걸기

"CoT 를 붙여도 별로 안 달라진다"가 이 절의 관찰이었습니다. 그런데 그것이 **CoT 가 쓸모없다**는 뜻인지, **이 모델이 규격을 스스로 잘 지킨다**는 뜻인지는 아직 모릅니다. 가리는 방법은 하나뿐입니다. **모델만 바꿔 다시 재는 것.**

- 아래 제공 셀이 더 작은 모델을 준비해 둡니다. 문서도 프롬프트도 서식도 **본문과 똑같이** 씁니다. 바뀌는 것은 모델 하나뿐입니다.
- 같은 문서(`doc`)에 **일반 프롬프트**를 걸어 `small_plain` 에, **CoT 프롬프트**를 걸어 `small_cot` 에 담으세요.
- 두 결과를 `(주어[타입], 관계, 목적어[타입])` 꼴로 **나란히** 출력하세요.

**확인 기준**: 교안_01 2-2 의 **관계 시그니처 표를 옆에 놓고 한 줄씩 대조**하세요. 볼 것은 건수가 아니라 **타입이 어긋난 줄**입니다. 교재가 세 번 돌렸을 때 이런 줄들이 나왔습니다.

- `(CYP2D6[Gene], ASSOCIATES, dementia[Disease])` : `ASSOCIATES` 는 (Disease -> Gene) 인데 **주어와 목적어가 뒤집혔습니다.**
- `(simvastatin[Compound], PALLIATES, myopathy[Symptom])` : `PALLIATES` 의 목적어는 `Disease` 인데 **`Symptom` 이 왔습니다.**
- `(VKORC1[Gene], ASSOCIATES, phenprocoumon[Compound])` : 주어도 목적어도 어긋났습니다.

그런 줄을 **일반 쪽과 CoT 쪽에서 각각 세어** 보세요. 교재가 돌렸을 때는 일반이 1~4건, CoT 가 매번 1건이었습니다. **본문에서 큰 모델로 돌린 결과와도 견주세요.** 큰 모델은 CoT 없이도 이런 줄이 거의 나오지 않았습니다. 건수는 실행마다 다르니 개수를 맞추려 하지 말고 **어느 쪽에 더 많은지**만 보면 됩니다.

In [ ]:
# [제공 코드] 더 작은 모델: 실행만 하세요. 문서·프롬프트·서식은 본문 그대로다
# 변수를 하나만 바꿔야 원인을 그 하나로 돌릴 수 있다. 여기서 바뀌는 것은 모델뿐이다
small = ChatOpenAI(model="gpt-4o-mini")
print("작은 모델 준비 완료")

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) build_extraction_prompt() 와 small 을 이어 붙여 doc 으로 뽑아 small_plain 에 담는다
# 2) build_cot_prompt() 와 small 을 이어 붙여 같은 doc 으로 뽑아 small_cot 에 담는다
# 3) 두 결과를 (주어[타입], 관계, 목적어[타입]) 꼴로 나란히 출력한다

### ✅ 바로 확인 퀴즈

**1.** 추출 서식에서 `reasoning` 칸을 `triples` 칸 **앞**에 두는 이유는?

<details><summary>정답 보기</summary>

모델은 서식의 칸을 **차례로** 채웁니다. `reasoning` 이 앞에 있으면 트리플을 적기 **전에** 개체 나열과 시그니처 대조를 하게 되고, 그 생각이 뒤 칸의 답에 반영됩니다. 뒤에 두면 이미 답을 쓴 뒤에 이유를 붙이는 셈이라 효과가 없습니다.

</details>

**2.** CoT 를 붙였더니 트리플 수가 달라졌습니다. 개수만 보고 "좋아졌다"거나 "나빠졌다"고 할 수 있을까요?

<details><summary>정답 보기</summary>

없습니다. 늘어난 것이 **맞는 트리플**이면 좋아진 것이고 **규격 밖 트리플**이면 나빠진 것입니다. 줄어든 쪽도 마찬가지로 무엇이 사라졌는지에 달렸습니다. 어느 쪽인지는 달라진 트리플을 하나씩 원문과 대조해 봐야 압니다. 개수만 보고 판단하면 안 되고, 이 판단을 지표로 만드는 것이 다음 단원의 주제입니다.

</details>

**3.** 데모 문서에서는 CoT 가 결과를 거의 바꾸지 않았습니다. 그러면 그 문서에서 CoT 는 쓸모가 없었던 걸까요?

<details><summary>정답 보기</summary>

결과만 보면 그렇지만, **판정 근거를 확인했다**는 소득이 있습니다. `reasoning` 3단계에 "대사·수송 관계는 BINDS" 라고 적혀 있어, 모델이 우리가 정한 판정 기준을 실제로 읽고 따랐음을 확인할 수 있습니다. 결과가 같아도 **왜 같은지**를 알게 된 것은 다릅니다. 다만 그 확인에 토큰과 시간을 쓴 것도 사실이라, 매번 붙일 이유는 되지 못합니다.

</details>

---
# 3. 그래프에 쌓기

여기까지는 트리플을 **뽑는** 일이었습니다. 여기서는 뽑은 것을 **그래프에 넣습니다.** 다만 모델이 준 값을 그대로 믿지 않고, 넣기 전에 거르고 넣을 때 두 가지를 더 남깁니다.

- **3-1** 시그니처로 거르고 id 로 `MERGE` 하며, 못 붙는 이름은 `:Candidate` 로 격리합니다.
- **3-2** 관계에 `evidence_level` 을 남겨 근거의 무게를 구분합니다.
- **3-3** 타입 계층으로 시그니처를 안 고치고 받아들일 개체만 넓힙니다.

## 3-1. 적재: 이름이 아니라 id 로

트리플을 뽑았으면 이제 **그래프에 쌓아** 쿼리로 꺼낼 수 있어야 합니다. 다만 모델이 돌려준 값을 그대로 믿으면 안 됩니다. 적재 전에 **세 번 거릅니다.**

<img src="images/three_filters.png" width="820">

**1번 후처리**는 개체 추출 단원에서 한 것(유형 검증 → 원문 등장 확인 → 중복 제거)을 트리플에 옮긴 것입니다. 그때는 개체 이름을 원문과 맞췄는데, 트리플에는 `evidence` 칸이 있어 **근거까지** 맞출 수 있습니다.

### 문법: 적재할 때 관계에 붙이는 속성 두 개
트리플 여섯 칸은 모델이 채웁니다. 적재 코드는 여기에 값 두 개를 더해, `MERGE` 로 만든 **관계(엣지)의 속성**으로 답니다. 노드가 아니라 관계에 붙습니다.

| 속성 | 값을 어디서 가져오나 | 왜 붙이나 |
|---|---|---|
| `source_doc_id` | 지금 읽고 있는 문서의 `doc_id` | 그 문서가 고쳐지거나 폐기되면 **그 문서에서 온 관계만** 골라 지울 수 있다 |
| `evidence_level` | 적재 코드가 `"reported"` 로 채운다 | 논문이 보고한 것인지 큐레이션 자료에 있는 것인지 구분한다(3-2에서 `curated` 와 견줍니다) |

**두 값은 모델에게 물을 수 없습니다.** 모델은 자기가 어떤 파일을 읽고 있는지 모르고, 그 사실이 데이터베이스에 이미 있는지도 모릅니다. 그래서 코드가 붙입니다.

In [ ]:
# model_dump() 로 객체를 dict 로 바꾸고, | 로 두 칸을 덧붙인다(원본은 그대로 둔다)
# 두 칸 다 모델이 정하는 값이 아니다. 어느 문서에서 왔는지는 파이프라인만 안다
rows = [tp.model_dump() | {"source_doc_id": demo["doc_id"], "evidence_level": "reported"}
        for tp in doc_result.triples]

print("칸 수:", len(rows[0]), "(트리플 여섯 칸 + 두 칸)")

In [ ]:
# 세로 막대 뒤 두 값이 방금 덧붙인 칸이다. 모델이 준 값이 아니라 파이프라인이 채운 값이다
for r in rows[:2]:
    print(r["subject"], r["relation"], r["object"], "|", r["source_doc_id"], r["evidence_level"])

### 문법: 후처리 - 근거로 걸러내기
모델이 지어낼 수 있는 것은 개체 이름만이 아닙니다. **근거 문장도 지어냅니다.** 그리고 근거가 진짜여도 **엉뚱한 문장**을 갖다 붙일 수 있습니다. 둘을 따로 봅니다.

In [ ]:
def ground_check(row, text):
    """근거가 원문에 있고 그 근거가 이 트리플을 뒷받침하는지 본다. 통과하면 None, 아니면 이유."""
    # 1) 코퍼스 전체가 아니라 '그 트리플이 나왔다고 적힌 그 문서' 하고만 대조한다
    if row["evidence"] not in text:
        return "근거가 원문에 없다"
    # 2) 근거가 진짜여도 다른 문장을 갖다 붙였을 수 있다. 주어와 목적어가 그 문장 안에 있어야 뒷받침이다
    for side in ("subject", "object"):
        if row[side] not in row["evidence"]:
            return f"근거에 {row[side]} 가 없다"
    return None


def drop_duplicates(rows):
    """같은 (주어, 관계, 목적어) 는 처음 만난 것 하나만 남긴다."""
    seen, kept = set(), []
    for r in rows:
        key = (r["subject"], r["relation"], r["object"])
        if key not in seen:
            seen.add(key)
            kept.append(r)
    return kept

In [ ]:
# 데모 문서에 걸어 본다. 몇 건이 빠졌나보다 '무엇이 왜' 빠졌나가 이 단계의 교훈이다
for r in rows:
    why = ground_check(r, demo["text"])
    mark = "통과" if why is None else f"버림({why})"
    print(f"{mark:<28} ({r['subject']}, {r['relation']}, {r['object']})")

> 이 검사는 **문자열이 정확히 일치해야 통과시킵니다.** 모델이 근거를 요약하거나 표기를 한 글자라도 고치면, 맞는 트리플도 함께 버려집니다. 그래서 프롬프트의 `- 근거(evidence)는 원문을 200자 이내로 그대로 인용한다` 와 서식의 `Field(description="주어. 논문에 적힌 표기 그대로")` 가 장식이 아닙니다. 그 두 줄이 모델에게 원문을 고치지 말라고 지시해서, 이 검사가 맞는 트리플을 버리지 않게 합니다.
>
> 검사를 만들 때는 **무엇이 걸리는지**만이 아니라 **맞는 것이 잘못 걸리지는 않는지**도 함께 봐야 합니다.

### 문법: 시그니처 검사
관계 이름만 보면 절반만 보는 것입니다. **주어·목적어 타입**까지 맞아야 합니다. 예를 들어 `gabapentin PALLIATES neuropathic symptoms` 는 관계 이름이 허용 목록에 있는데도 규격 밖입니다. 목적어가 `Symptom` 이라, 목적어를 `Disease` 로 못 박은 `PALLIATES` 의 시그니처와 어긋나기 때문입니다. 서식은 이런 **칸 사이의 어긋남**을 못 봅니다.

In [ ]:
def check_signature(row):
    """트리플이 온톨로지의 시그니처를 지키는지 본다. 통과하면 None, 아니면 이유 문자열."""
    # 참·거짓이 아니라 '이유' 를 돌려준다. 무엇이 왜 기각됐는지 남겨야 프롬프트를 고칠 수 있다
    if row["relation"] not in RELATION_SIGNATURES:
        return "허용 관계가 아님"          # 온톨로지에 없는 관계 이름을 지어낸 경우
    subj_type, obj_type, _ = RELATION_SIGNATURES[row["relation"]]   # 판정 기준(셋째 칸)은 여기서 안 쓴다
    if row["subject_type"] != subj_type:
        return f"주어 타입이 {subj_type} 이어야 함"
    # 목적어 타입까지 봐야 방향이 뒤집힌 트리플을 잡는다. 관계 이름만 보면 절반만 보는 것이다
    if row["object_type"] != obj_type:
        return f"목적어 타입이 {obj_type} 이어야 함"
    return None


# 기각 사유를 함께 찍어야 어느 트리플이 왜 걸렸는지 눈으로 확인할 수 있다
for r in rows:
    reason = check_signature(r)
    mark = "통과" if reason is None else f"기각({reason})"
    print(f"{mark:<28} ({r['subject']}, {r['relation']}, {r['object']})")

### 문법: 이름 해소와 격리
지난 시간의 `lookup_id` 가 여기서 쓰입니다. `(id, 사유)` 두 칸을 돌려주는데 여기서는 첫 칸만 봅니다. id 가 나오면 그 id 로, 안 나오면 `:Candidate` 로 보냅니다. 사전이 이름을 모르든(`miss`) 후보가 둘이라 못 고르든(`ambiguous`) 오늘 할 일은 격리로 같기 때문입니다. `:Candidate` 노드는 **이름과 타입 둘을 함께 키로** 씁니다. 이름 하나만 키로 쓰면 격리한 자리에서 다시 같은 사고가 나기 때문입니다.

In [ ]:
def node_pattern(alias, name, node_type, prefix):
    """노드 하나의 MERGE 패턴과 파라미터를 만든다. (패턴, 파라미터, 격리 여부)를 돌려준다."""
    # alias 는 Cypher 안에서 이 노드를 부를 이름(a·b), prefix 는 파라미터 이름이 겹치지 않게 붙이는 머리글자(s·o)
    found = lookup_id(name, node_type)[0]   # 두 칸 중 첫 칸(id)만 쓴다. 못 붙으면 None 이다
    if found:
        return f"({alias}:{node_type} {{id: ${prefix}_id}})", {f"{prefix}_id": found}, False
    # 사전에 없거나 애매한 이름. 이름과 타입 둘을 함께 키로 삼아 따로 세운다
    return (f"({alias}:Candidate {{name: ${prefix}_name, type: ${prefix}_type}})",
            {f"{prefix}_name": name, f"{prefix}_type": node_type}, True)


print(node_pattern("a", "clopidogrel", "Compound", "s"))
print(node_pattern("a", "Tapinarof", "Compound", "s"))

### 문법: 값은 파라미터로 넘긴다
레이블(`:Compound`)과 관계 타입(`:BINDS`)은 **문장의 구조**라서 문자열에 끼워 넣어야 합니다. 하지만 **값**은 다릅니다. 근거 문장에는 `Parkinson's disease` 처럼 **작은따옴표**가 들어 있어, 문장에 그대로 끼워 넣으면 Cypher 가 그 자리에서 깨집니다.

그래서 값은 `$이름` 자리표시자로 두고 `run_cypher(cy, 이름=값)` 으로 따로 넘깁니다. 이것이 실무 관례이기도 합니다(값을 문장에 붙이면 주입 공격에도 열립니다).

In [ ]:
def load_triple(row):
    """트리플 한 줄을 그래프에 MERGE 한다. 격리한 노드가 있으면 True 를 돌려준다."""
    # 주어와 목적어를 따로 해소한다. 한쪽만 격리되는 경우도 있다
    s_pat, s_params, s_new = node_pattern("a", row["subject"], row["subject_type"], "s")
    o_pat, o_params, o_new = node_pattern("b", row["object"], row["object_type"], "o")
    # 노드 둘을 먼저 세우고 그 사이를 잇는다. 세 줄 다 MERGE 라 같은 트리플을 두 번 넣어도 늘지 않는다
    cy = (f"MERGE {s_pat}\n"
          f"MERGE {o_pat}\n"
          f"MERGE (a)-[r:{row['relation']}]->(b)\n"
          # 이름은 이미 있으면 덮지 않는다. 큐레이션 데이터가 정한 표기를 논문 표기로 밀어내지 않기 위해서다
          "SET a.name = coalesce(a.name, $sname), b.name = coalesce(b.name, $oname),\n"
          # 근거와 출처도 coalesce 다. 같은 사실을 다른 논문이 또 말해도 처음 근거를 밀어내지 않는다
          "    r.evidence = coalesce(r.evidence, $ev),\n"
          "    r.source_doc_id = coalesce(r.source_doc_id, $doc),\n"
          # 등급도 coalesce 다. 이미 curated 인 관계를 논문 한 편의 reported 로 끌어내리지 않는다
          "    r.evidence_level = coalesce(r.evidence_level, $level)")
    run_cypher(cy, sname=row["subject"], oname=row["object"], ev=row["evidence"],
               doc=row["source_doc_id"], level=row["evidence_level"], **s_params, **o_params)
    return s_new or o_new   # 한쪽이라도 격리됐으면 True. 바깥에서 격리 건수를 센다

In [ ]:
# 데모 문서의 트리플을 적재한다: 후처리와 시그니처를 통과한 것만, 이름은 id 로 바꿔서
loaded, dropped, rejected, isolated = 0, 0, 0, 0
for r in drop_duplicates(rows):
    if ground_check(r, demo["text"]) is not None:
        dropped += 1
        continue                 # 근거가 뒷받침하지 않는 트리플은 넣지 않는다
    if check_signature(r) is not None:
        rejected += 1
        continue                 # 규격 밖 트리플은 그래프에 넣지 않고 세기만 한다
    if load_triple(r):
        isolated += 1            # 적재는 됐지만 한쪽 끝이 :Candidate 로 격리된 경우
    loaded += 1
print("적재:", loaded, "/ 근거 미달:", dropped, "/ 기각:", rejected, "/ 격리:", isolated)

In [ ]:
# 근거 등급이 붙은 관계만 본다. 같은 DB 에 다른 실습 그래프가 있어도 섞이지 않는다
for row in run_cypher("MATCH (a)-[r]->(b) WHERE r.evidence_level IS NOT NULL "
                      "RETURN a.name AS 주어, type(r) AS 관계, b.name AS 목적어, "
                      "       r.evidence_level AS 근거등급 "
                      "ORDER BY 관계, 주어, 목적어"):
    print(row)

> 뽑은 트리플이 그대로 들어갔습니다. **기각도 격리도 0 입니다.** 이 문서에서는 관계가 전부 규격에 맞았고 이름도 전부 사전에 있었습니다.
>
> 그러면 검사 셋은 쓸모가 없었을까요? **논문 두 편을 더 넣어 보면 답이 나옵니다.**

### 논문을 두 편 더 넣어 본다
- **PMC13461326**: 유방암 수술 후 수면장애의 약물 관리 리뷰
- **PMC13494208**: AHR 이라는 수용체의 약리학 리뷰. **최근에 승인된 약**이 여럿 나오는데 우리 사전은 2016년 시점이라 그 이름들을 모릅니다.

두 편은 **한 번에** 뽑습니다. `extraction_chain.batch([...])` 로 묶으면 문서 수만큼 `invoke` 를 부르는 것보다 빠르고, 결과는 **입력 순서 그대로** 돌아옵니다. 다만 그대로 두면 문서 수만큼 요청이 한꺼번에 나가 **API 분당 요청 한도에 걸릴 수 있으므로**, `config={"max_concurrency": 2}` 로 한 번에 내보낼 요청 수를 정해 둡니다. 뽑은 뒤 검사와 적재는 문서마다 따로 돕니다. 같은 `check_signature` 와 같은 `load_triple` 에 문서만 갈아 끼웁니다. 기각된 트리플은 **이유까지 함께** 찍어 봅니다.

In [ ]:
# 논문 두 편을 더 넣는다. 검사·적재 코드는 그대로 두고 문서만 갈아 끼운다
more_ids = ['PMC13461326', 'PMC13494208']


# 모델 호출은 문서마다 따로 부르지 않고 한 번에 묶는다
# max_concurrency 는 한 번에 내보낼 요청 수다. 이게 없으면 문서 수만큼 한꺼번에 나간다
more_results = extraction_chain.batch([{"text": core[d]["text"]} for d in more_ids],
                                      config={"max_concurrency": 2})

# 결과가 입력 순서 그대로 오므로 zip 으로 짝지어도 어긋나지 않는다
for doc_id, result in zip(more_ids, more_results):
    row = core[doc_id]
    # 적재할 때 붙이는 두 칸은 문서가 바뀌어도 같은 자리에서 붙는다
    more_rows = [tp.model_dump() | {"source_doc_id": doc_id, "evidence_level": "reported"}
                 for tp in result.triples]
    ok, bad, no, iso = 0, 0, 0, 0     # 문서마다 네 가지를 따로 센다
    for r in drop_duplicates(more_rows):
        reason = ground_check(r, row["text"])
        if reason is not None:
            bad += 1
            print(f"  근거 미달({reason}): ({r['subject']}, {r['relation']}, {r['object']})")
            continue
        reason = check_signature(r)
        if reason is not None:
            no += 1
            print(f"  기각({reason}): ({r['subject']}, {r['relation']}, {r['object']})")
            continue
        if load_triple(r):
            iso += 1
        ok += 1
    print(f"{doc_id}: 적재 {ok} / 근거 미달 {bad} / 기각 {no} / 격리 {iso}")

> **세 검사는 서로 다른 것을 잡습니다.** 위 출력의 `적재 / 근거 미달 / 기각 / 격리` 네 숫자에서 건수는 실행마다 다르지만 갈래는 늘 같습니다.
>
> - **근거 미달**: 근거가 원문에 없거나, 근거는 진짜인데 그 트리플을 뒷받침하지 않는다.
> - **기각**(시그니처 검사): 관계 이름은 허용 목록에 있는데 타입이 어긋난다. `gabapentin PALLIATES neuropathic symptoms` 는 목적어가 `Symptom` 인데 `PALLIATES` 의 목적어는 `Disease` 라고 적어 두었습니다.
> - **격리**(이름 해소): 규격은 통과했는데 한쪽 끝이 사전에 없다. `H1 receptor` 처럼 정식 유전자 기호가 아닌 표기가 여기로 갑니다.
>
> 어느 수가 0 이면 이번 실행에서 그 갈래가 안 나온 것이지 그 검사가 안 돈 것이 아닙니다.

In [ ]:
# Hetionet 노드와 섞이지 않고 따로 서 있다
for row in run_cypher("MATCH (c:Candidate) "
                      "RETURN c.name AS 이름, c.type AS 뽑힌타입 ORDER BY 이름"):
    print(row)

> **왜 못 붙었는지가 넷으로 갈립니다.** 이름만 보면 다 똑같이 "없다"인데 처방이 다릅니다.
>
> | 이름 | 왜 못 붙었나 | 처방 |
> |---|---|---|
> | `Tapinarof`·`Laquinimod` | 승인이 늦거나 시험 단계인 약이라 2016년 사전에 아예 없다 | 사전을 넓혀야 한다 |
> | `Parkinson disease` | 사전에는 `Parkinson's disease` 로 있다. 아포스트로피 한 글자 차이 | 표기를 맞추면 붙는다 |
> | `Huntington disease` | 사전에 **있다.** 다만 `Symptom` 으로 있어 `Disease` 로 물으면 안 붙는다 | 타입까지 맞아야 같은 개체다 |
> | `H1 receptor` | 유전자 기호가 아니라 단백질 통칭이다(기호는 `HRH1`) | 층위를 잇는 매핑 규칙이 필요하다 |
>
> 셋째 줄만 성격이 다릅니다. 이름이 사전에 **있는데도** `lookup_id` 의 `entry["label"] == node_type` 검사에서 걸립니다.
>
> **지금 아무것도 잃지 않았습니다.** 붙일 곳을 못 찾은 사실도 근거와 출처와 함께 그래프에 남아 있습니다. 표기를 맞추는 일은 엔티티 정규화 단원이, `:Candidate` 를 정식 노드로 승격하는 일은 증분 적재 단원이 맡습니다.

### 🖐️ 함께 따라하기: 격리된 노드에 걸린 관계 세기

`:Candidate` 노드에 걸린 관계가 몇 건인지 세어 보세요. 한쪽 끝이라도 `:Candidate` 면 셉니다.

- Cypher: `MATCH (c:Candidate)-[r]-() RETURN count(DISTINCT r) AS 관계수`
- 이어서 이 실습이 만든 관계 수도 출력해 견줘 보세요. `MATCH ()-[r]->() WHERE r.evidence_level IS NOT NULL RETURN count(r) AS 관계수`

**확인 기준**: 두 번째 값보다 작고 0 보다 큰 수가 나오면 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (c:Candidate)-[r]-() RETURN count(DISTINCT r) AS 관계수 를 run_cypher 로 실행한다
# 2) MATCH ()-[r]->() WHERE r.evidence_level IS NOT NULL RETURN count(r) AS 관계수 도 실행한다
# 3) 두 값을 나란히 출력한다

### ✅ 바로 확인 퀴즈

**1.** 적재 루프가 관계 이름만 보지 않고 **주어·목적어 타입까지** 대조하는 이유는?

<details><summary>정답 보기</summary>

관계 이름이 맞아도 **주어·목적어 타입이 어긋날 수 있기** 때문입니다. 앞에서 기각된 `gabapentin PALLIATES neuropathic symptoms` 가 그렇습니다. 이름은 허용 목록에 있지만 목적어가 `Symptom` 이라, 목적어를 `Disease` 로 못 박은 `PALLIATES` 의 시그니처와 어긋납니다. 타입까지 대조해야 이런 트리플을 그래프에 들어오기 전에 잡습니다.

</details>

**2.** 근거 문장을 Cypher 문자열에 직접 끼워 넣지 않고 **파라미터**로 넘기는 이유는?

<details><summary>정답 보기</summary>

근거는 논문 원문이라 작은따옴표(`Parkinson's`)가 들어 있습니다. 문장에 그대로 끼우면 문자열이 그 자리에서 끝나 **Cypher 가 깨집니다.** 값을 파라미터로 넘기면 따옴표가 있어도 안전하고, 남이 넣은 문자열로 쿼리가 바뀌는 사고도 막습니다.

</details>

## 3-2. 근거 등급: 이 사실은 어디서 왔는가

방금 넣은 논문에서 뽑힌 트리플 중에 `Tapinarof -[:TREATS]-> plaque psoriasis` 와 `Laquinimod -[:TREATS]-> multiple sclerosis` 가 있습니다. 화살표만 보면 똑같습니다. 그런데 원문은 이렇게 말합니다.

| 트리플 | 원문이 한 말 |
|---|---|
| Tapinarof TREATS plaque psoriasis | FDA 가 **2022년에 승인**했다 |
| Laquinimod TREATS multiple sclerosis | 경구 치료제로 **연구되었다**(was investigated) |

> 이번 실행에서 이 두 줄이 안 나왔다면 추출기가 그 문장을 안 집었다는 뜻입니다. 아래 근거 등급 이야기는 그대로 읽어도 됩니다.

뒤엣것은 **효능이 입증됐다는 말이 아닙니다.** 라퀴니모드는 임상시험을 거쳤지만 승인된 치료제가 되지는 못했습니다. 그런데 추출기는 둘을 똑같이 `TREATS` 로 적어 왔습니다.

이 그래프가 나중에 질문에 답하는 데 쓰인다면(뒤 단원의 GraphRAG), "이 병에 무슨 약을 쓰나"라는 질문에 두 답이 **같은 무게로** 나옵니다. 의료에서 이건 안전 문제입니다.

In [ ]:
# 화살표만 보면 다 같은 TREATS 다. 무게가 다르다는 것은 근거 원문에만 적혀 있다
# 모델이 논문 표기 그대로 쓰므로 이름의 대소문자가 실행마다 다르다. toLower 로 눌러 찾는다
for row in run_cypher("MATCH (a)-[r:TREATS]->(b) "
                      "WHERE toLower(a.name) IN ['tapinarof', 'laquinimod'] "
                      "RETURN a.name AS 약, b.name AS 병, r.evidence AS 근거 "
                      "ORDER BY 약, 병"):
    print(row["약"], "->", row["병"])
    print("   ", row["근거"][:150])

### 문법: 관계에 등급을 남긴다
적재 함수는 관계에 `evidence`·`source_doc_id`·`evidence_level` 세 속성을 붙였습니다. **관계에 속성을 붙일 수 있다는 것이 LPG 의 특권입니다.** 그래프DB 단원에서 본 RDF 였다면 이 셋을 담으려고 관계를 다시 노드로 풀어야 했습니다.

완전한 해결책은 아니지만, 최소한 **어디서 온 사실인지**는 남길 수 있습니다. 관계 속성 `evidence_level` 에 두 값 중 하나를 둡니다.

| 값 | 뜻 | 어디서 |
|---|---|---|
| `curated` | 사람이 여러 근거를 모아 정리한 데이터베이스에 들어 있다 | Hetionet 같은 큐레이션 자료 |
| `reported` | 논문 한 편이 그렇게 보고했다 | 오늘 우리가 뽑은 트리플 |

규칙은 두 줄입니다.

- **큐레이션 자료는 등급을 확정한다**: `SET r.evidence_level = 'curated'`
- **논문은 등급을 낮추지 않는다**: `SET r.evidence_level = coalesce(r.evidence_level, 'reported')`

`coalesce(a, b)` 는 "a 가 있으면 a, 없으면 b"입니다. 그래서 이미 `curated` 인 관계에 논문 근거가 더 붙어도 등급은 그대로 남습니다.

근거 문장(`evidence`)과 출처(`source_doc_id`)에도 같은 이유로 `coalesce` 를 씁니다. 같은 사실을 다른 논문이 또 말해도 **처음 적어 둔 근거를 밀어내지 않습니다.** 그러면 "이 문서에서 온 트리플만" 걷어내는 일이 계속 가능합니다. 근거를 여러 개 **쌓아 두는** 방법은 증분 적재 단원에서 다룹니다.

이 값이 왜 중요한지는 마지막 단원에서 드러납니다. **GraphRAG 답변이 이 등급을 표시하지 못하면 그 시스템은 쓸 수 없다**는 것이 이 과정의 합격 기준 중 하나입니다.

### 큐레이션 자료를 얹어 보기
지식그래프를 적재한 단원에서 쓴 Hetionet 에서, **오늘 논문에 나온 약물·유전자 주변의 관계만** 골라 둔 파일이 `data/hetionet_curated.jsonl` 입니다. 이 관계들은 이름이 아니라 **id 로** 적혀 있어 해소 단계가 필요 없습니다.

In [ ]:
# [제공 코드] 큐레이션 관계 읽기: 실행만 하세요
curated = []
for line in Path("data/hetionet_curated.jsonl").read_text(encoding="utf-8").splitlines():
    curated.append(json.loads(line))
print("큐레이션 관계:", len(curated), "건")

In [ ]:
# [제공 코드] (이어서)
print(curated[0])

In [ ]:
def load_curated(row):
    """큐레이션 관계 한 줄을 적재한다. 등급을 curated 로 확정하고 이름도 이쪽 표기로 맞춘다."""
    # 이름 해소도 시그니처 검사도 없다. 이미 id 와 타입이 정리된 자료라 그대로 MERGE 한다
    cy = (f"MERGE (a:{row['subject_type']} {{id: $sid}})\n"
          f"MERGE (b:{row['object_type']} {{id: $oid}})\n"
          f"MERGE (a)-[r:{row['relation']}]->(b)\n"
          # 큐레이션 자료가 표기와 등급을 확정한다. coalesce 없이 덮어쓴다
          "SET a.name = $sname, b.name = $oname,\n"
          "    r.evidence_level = 'curated', r.source = 'Hetionet v1.0'")
    run_cypher(cy, sid=row["subject_id"], oid=row["object_id"],
               sname=row["subject"], oname=row["object"])


# 이미 논문에서 들어간 관계와 겹치면 MERGE 라 새로 생기지 않고 등급만 curated 로 올라간다
for row in curated:
    load_curated(row)
print("큐레이션 관계 적재 완료")

In [ ]:
# curated 와 reported 가 각각 몇 건인지가 이 절의 관찰 대상이다
for row in run_cypher("MATCH ()-[r]->() WHERE r.evidence_level IS NOT NULL "
                      "RETURN r.evidence_level AS 등급, count(r) AS 건수 ORDER BY 등급"):
    print(row)

In [ ]:
# 두 속성이 다 있다 = 논문에서도 뽑혔고 큐레이션 자료에도 있던 관계다
for row in run_cypher("MATCH (a)-[r]->(b) "
                      "WHERE r.source_doc_id IS NOT NULL AND r.source IS NOT NULL "
                      "RETURN a.name AS 주어, type(r) AS 관계, b.name AS 목적어, "
                      "       r.evidence_level AS 등급, r.source_doc_id AS 논문 "
                      "ORDER BY 주어"):
    print(row)

> 데모 문서에서 뽑은 트리플은 **전부 큐레이션 자료에도 있어** 등급이 `curated` 로 올라갔고, 논문 근거(`source_doc_id`)는 그대로 남았습니다. 2016년에 정리된 내용이 2026년 논문에서도 확인됐다는 기록이 됩니다. 두 번째 문서 것은 `reported` 로 남았습니다. 큐레이션 자료에 아직 없는, 논문 한 편이 말한 사실입니다.
>
> **다만 이 소단원을 연 두 줄은 아직 안 갈립니다.** `Tapinarof TREATS plaque psoriasis`(FDA 승인)와 `Laquinimod TREATS multiple sclerosis`(연구 단계)가 둘 다 큐레이션 자료에 없어 똑같이 `reported` 입니다. `evidence_level` 이 가르는 것은 **승인이냐 시험이냐가 아니라 어디서 온 사실이냐**입니다. 말의 세기까지 담으려면 칸이 하나 더 필요하고, 그건 다음 시간 품질 측정의 주제입니다.

### 🖐️ 함께 따라하기: TREATS 와 PALLIATES 를 갈라 조회하기

큐레이션 자료에는 `Tramadol -[:PALLIATES]-> restless legs syndrome` 이 들어 있습니다. 트라마돌은 하지불안증후군을 **낫게 하는 약이 아니라 증상을 완화하는 약**이라 이렇게 적혀 있는 것입니다.

`TREATS` 관계와 `PALLIATES` 관계를 **따로** 조회해 출력하세요.

- 관계 종류를 한 자리에 둘 넣는 문법은 `-[r:TREATS|PALLIATES]->` 입니다.
- `주어 / 관계 / 목적어 / 등급` 을 관계 종류 순으로 정렬해 출력하세요.

**확인 기준**: `PALLIATES` 는 큐레이션 자료가 넣은 세 건이 `curated` 로 나옵니다. 논문에서 `PALLIATES` 가 더 뽑혔다면 `reported` 가 얹혀 네 건이 될 수도 있어요. `TREATS` 는 시드의 `curated` 에 이번에 뽑힌 `reported` 가 얹혀 **두 등급이 섞이는 것이 보통**인데, 이번 실행에서 `TREATS` 가 하나도 안 뽑혔다면 `curated` 만 나옵니다. 둘 다 정상이에요. 볼 것은 **한 관계 안에 등급이 섞일 수 있다**는 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (a)-[r:TREATS|PALLIATES]->(b) 로 두 관계를 한 번에 잡는다
# 2) a.name, type(r), b.name, r.evidence_level 을 돌려받는다
# 3) 관계 종류 순으로 정렬해 한 줄씩 출력한다

> 결과에서 **같은 약이 두 줄로 나온 것**을 찾아보세요. 교재가 돌렸을 때는 `Carbidopa` 가 그랬습니다.
>
> | 주어 | 관계 | 목적어 | 등급 |
> |---|---|---|---|
> | Carbidopa | `PALLIATES` | Parkinson's disease | curated |
> | Carbidopa | `TREATS` | Parkinson disease | reported |
>
> 같은 약과 같은 병인데 그래프에는 **따로** 들어가 있습니다(위쪽은 큐레이션 자료가, 아래쪽은 논문 추출이 넣은 것입니다. 이번 실행에서 논문 쪽이 안 뽑혔다면 아래 줄은 없습니다). 두 가지가 겹쳤습니다.
>
> 1. 목적어 이름이 `Parkinson's disease` 와 `Parkinson disease` 로 **아포스트로피 한 글자** 다릅니다. 그래서 아래쪽 목적어는 `:Candidate` 로 격리됐고 위쪽 노드와 붙지 못했습니다.
> 2. 관계도 다릅니다. 큐레이션 자료는 카비도파가 파킨슨병을 **낫게 하는 약이 아니라 증상을 다스리는 약**이라고 보아 `PALLIATES` 로 적어 두었습니다. 논문은 "used to treat" 라고 썼고 추출기는 그 말을 `TREATS` 로 옮겼습니다.
>
> 첫째는 **표기 정규화**의 문제이고 둘째는 **판정 기준**의 문제입니다. 앞엣것은 엔티티 정규화 단원에서, 뒤엣것은 바로 다음 시간의 품질 측정에서 다룹니다. 지금 할 일은 둘을 억지로 합치지 않고 **왜 갈렸는지 알아볼 수 있게 남겨 두는 것**입니다.

### ✅ 바로 확인 퀴즈

**1.** 논문에서 뽑은 관계를 적재할 때 `r.evidence_level = 'reported'` 라고 **덮어쓰지 않고** `coalesce(r.evidence_level, 'reported')` 를 쓰는 이유는?

<details><summary>정답 보기</summary>

그 관계가 이미 `curated` 일 수 있기 때문입니다. 덮어쓰면 **여러 근거로 정리된 사실이 논문 한 편의 보고로 격하됩니다.** `coalesce` 는 값이 있으면 그대로 두므로 등급이 내려가지 않습니다.

</details>

**2.** 어떤 관계의 `evidence_level` 이 `reported` 입니다. 이 관계를 그래프에서 지워야 할까요?

<details><summary>정답 보기</summary>

아닙니다. `reported` 는 **틀렸다는 뜻이 아니라 근거가 논문 한 편이라는 뜻**입니다. 최신 논문의 사실은 대부분 여기서 시작합니다. 할 일은 지우는 것이 아니라 **답할 때 그 사실을 밝히는 것**입니다.

</details>

## 3-3. 타입 계층 얹기

2-1 에서 남겨 둔 숙제가 있습니다. 데모 문서에서 뽑은 트리플이 전부 `BINDS` 인데 성격이 갈렸습니다. `CYP2C19`·`CYP2D6` 같은 것은 약을 분해하는 **효소**, `SLCO1B1` 은 약을 나르는 **수송체**, `VKORC1` 은 약이 노리는 **표적**입니다. 관계 이름으로는 이미 구분을 포기했습니다.

그런데 **노드 쪽에서는 일부 되찾을 수 있습니다.** 관계가 아니라 **유전자 자체**에 "이건 효소다", "이건 수송체다"를 표시해 두는 것입니다. 그러면 "이 약이 붙은 유전자 중 효소인 것"까지는 물을 수 있게 됩니다.

### 문법: 하위 타입을 상위 레이블과 겹쳐 붙인다
**타입 계층**이 이 둘을 함께 얻게 해 줍니다. `Enzyme`·`Transporter` 를 `Gene` 의 **하위 타입**으로 선언하고, Neo4j 에서는 레이블을 **겹쳐** 붙입니다(`:Gene:Enzyme`). 그러면 상위 타입으로 물었을 때 하위도 함께 잡힙니다. 시그니처는 그대로 두고 받아들일 개체만 넓히는 것입니다.

<img src="images/type_hierarchy.png" width="700">

In [ ]:
# 온톨로지에 타입 계층을 한 줄 얹는다. 하위 타입 -> 상위 타입
# 한 단계만 둔다. 계층을 깊게 파면 관리 비용이 빠르게 는다
TYPE_HIERARCHY = {"Enzyme": "Gene", "Transporter": "Gene"}


def labels_for(node_type):
    """하위 타입이면 상위 타입 레이블까지 겹쳐 돌려준다 (Gene, Enzyme)."""
    # 상위가 없으면(=이미 최상위 타입) 자기 자신 하나만 돌려준다
    parent = TYPE_HIERARCHY.get(node_type)
    return [parent, node_type] if parent else [node_type]


print(labels_for("Enzyme"), labels_for("Compound"))

In [ ]:
def set_labels(node_id, node_type):
    """id 로 노드를 찾아 상위·하위 레이블을 함께 붙인다."""
    # 레이블 여러 개는 콜론으로 잇는다 -> SET n:Gene:Enzyme
    # SET 은 있던 레이블을 지우지 않는다. 겹쳐 붙이는 것이라 :Gene 쿼리에도 계속 잡힌다
    labels = ":".join(labels_for(node_type))
    run_cypher(f"MATCH (n {{id: $id}}) SET n:{labels}", id=node_id)


# 적재할 때는 타입이 Gene 이었지만, 실제로는 대사 효소라는 것을 알고 있으므로 하위 타입으로 다시 붙인다
set_labels("Gene::1557", "Enzyme")     # CYP2C19
set_labels("Gene::1565", "Enzyme")     # CYP2D6

# 레이블 수 내림차순이라 방금 겹쳐 붙인 둘이 맨 위로 온다
for row in run_cypher("MATCH (g:Gene) RETURN g.name AS 유전자, labels(g) AS 레이블 "
                      "ORDER BY size(labels(g)) DESC, 유전자 LIMIT 5"):
    print(row)

> `:Gene` 으로 물었는데 `:Gene:Enzyme` 인 CYP2C19 가 **함께** 잡힙니다. 레이블을 겹쳐 붙였기 때문입니다. 시그니처(`BINDS` 의 목적어 = Gene)는 한 글자도 안 고쳤는데 표현력만 늘었습니다.
>
> 계층을 깊게 파면 관리 비용이 빠르게 늘어납니다. **한 단계**로 시작해, 쿼리에서 실제로 구분이 필요할 때만 더합니다.

### 🖐️ 함께 따라하기: 수송체에 하위 타입 붙이기

SLCO1B1(`Gene::10599`)은 효소가 아니라 **수송체**입니다. `set_labels` 로 `"Transporter"` 하위 타입을 붙이고, `:Transporter` 로 물었을 때 잡히는지 확인하세요.

- 붙인 뒤 `MATCH (t:Transporter) RETURN t.name AS 유전자, labels(t) AS 레이블` 로 조회합니다.

**확인 기준**: SLCO1B1 이 나오고 레이블에 `Gene` 과 `Transporter` 가 모두 들어 있으면 성공입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) set_labels('Gene::10599', 'Transporter') 를 호출한다
# 2) MATCH (t:Transporter) RETURN t.name AS 유전자, labels(t) AS 레이블 로 조회해 출력한다

> **되찾은 것과 못 되찾은 것을 정확히 갈라 둡시다.**
>
> 되찾은 것: "이 약이 붙은 유전자 중 **효소인 것**만" 같은 쿼리가 가능해졌습니다. `MATCH (c:Compound)-[:BINDS]->(g:Enzyme)` 로 물으면 됩니다.
>
> 못 되찾은 것: **그 관계가 대사인지 표적 결합인지는 여전히 모릅니다.** 레이블은 유전자에 붙은 것이지 관계에 붙은 것이 아니기 때문입니다. 어떤 약에는 표적이고 어떤 약에는 대사 효소인 유전자도 있는데, 그런 경우는 노드 표시로 갈 수 없습니다.
>
> 이게 지난 시간 마지막에 말한 그 이야기입니다. **온톨로지가 구분하지 않기로 한 것은 나중에 완전히 되찾을 수 없습니다.** 우회로가 있어도 원래 자리만큼은 아닙니다. 그래서 관계를 정하는 자리가 그렇게 중요합니다.

### ✅ 바로 확인 퀴즈

**1.** `:Gene:Enzyme` 처럼 레이블을 **겹쳐** 붙이면 무엇이 좋아지나요?

<details><summary>정답 보기</summary>

상위 타입(`:Gene`)으로 물으면 하위(`:Enzyme`)도 함께 잡히고, 필요하면 `:Enzyme` 으로 좁혀 물을 수도 있습니다. 시그니처(목적어 = Gene)는 한 글자도 안 고치고 받아들일 개체만 넓힙니다.

</details>

**2.** 계층을 3단계, 4단계로 깊게 파면 무엇이 문제인가요?

<details><summary>정답 보기</summary>

계층이 깊어질수록 **관리 비용**이 빠르게 늘어납니다(타입이 늘고, 어디에 넣을지 판단이 어려워지고, 쿼리도 복잡해집니다). **한 단계**로 시작해 쿼리에서 실제로 구분이 필요할 때만 더합니다.

</details>

---
## 🚀 종합 클론코딩: 오늘 배운 것을 한 함수로

오늘 만든 조각을 전부 이어 붙입니다. 논문 발췌 **여러 편**을 받아 **배치 추출 → 후처리 → 시그니처 검사 → id 해소·격리 → 적재**까지 한 번에 하는 `extract_and_load_batch(rows)` 를 완성하세요.

| 단계 | 쓰는 것 | 어디서 만들었나 |
|---|---|---|
| 배치 추출 | `extraction_chain.batch` | 2-1 (온톨로지 주입 템플릿 + 값을 좁힌 서식), 3-1 (배치 호출) |
| 후처리 | `drop_duplicates` · `ground_check` | 3-1 |
| 시그니처 검사 | `check_signature` | 3-1 |
| id 해소·적재 | `load_triple` | 3-1 |

`rows` 는 `pgx_core.jsonl` 의 줄들(`doc_id`·`text` 를 가진 dict)의 리스트입니다. **모델 호출은 `batch` 로 한 번에 묶고**, 검사와 적재는 문서마다 따로 돌립니다. 돌려주는 값은 문서마다 `(doc_id, 적재, 근거 미달, 기각, 격리)` 다섯 칸인 리스트입니다. 새로 만들 것은 없습니다. 이미 만든 함수를 **순서대로 부르는 일**이 전부입니다.

완성한 뒤 아래 `batch_ids` 의 논문 세 편을 넣어 문서별 결과를 출력하세요. 세 편을 마치면 오늘 쓴 논문 **여섯 편이 전부** 그래프에 들어갑니다.

**확인 기준**: 돌려받은 리스트의 길이가 넣은 문서 수와 같고(`batch` 는 입력 순서 그대로 돌려줍니다), 문서마다 네 수가 찍히며, 그 넷을 더한 값이 그 문서에서 뽑힌 트리플 수와 같습니다(격리는 적재된 것 중 일부를 다시 센 것이라 더하지 않습니다). 건수는 실행마다 다르지만 이 셈은 늘 맞아떨어집니다.

In [ ]:
# 아직 적재하지 않은 나머지 논문 발췌 세 편. 앞에서 넣은 세 편과 겹치지 않는다
batch_ids = ['PMC13432136', 'PMC13473732', 'PMC13494139']
for doc_id in batch_ids:
    print(doc_id, "|", core[doc_id]["title"][:60])

In [ ]:
# 🚀 종합 (아래 순서대로 직접 작성해 보세요)
# 1) extract_and_load_batch(rows) 를 정의한다
# 2) extraction_chain.batch 로 rows 의 본문을 한 번에 넣어 결과를 받는다
#    config={"max_concurrency": 2} 를 함께 넘겨 한 번에 내보낼 요청 수를 둘로 제한한다
# 3) zip 으로 rows 와 결과를 짝지어 문서 하나씩 돈다
# 4) 뽑은 트리플에 source_doc_id 와 evidence_level='reported' 를 덧붙인다
# 5) drop_duplicates 로 같은 트리플을 하나로 줄인다
# 6) ground_check 를 통과 못 하면 근거 미달로 세고 넘어간다
# 7) check_signature 를 통과 못 하면 기각으로 세고 넘어간다
# 8) 남은 것만 load_triple 로 적재한다(돌려준 값이 True 면 격리도 함께 센다)
# 9) 문서마다 (doc_id, 적재, 근거 미달, 기각, 격리) 를 모아 return 하고, 돌면서 출력한다

---
## 이번 강의 정리

| 소단원 | 하는 일 | 핵심 |
|---|---|---|
| 1-1 온톨로지 | 시그니처를 dict 한 곳에 | `RELATION_SIGNATURES` = 진실의 원천 |
| 1-2 구조화 출력 | 트리플을 틀로 받기 | `with_structured_output(Extraction)` |
| 1-2 JSON Schema | 모델이 실제로 받는 규격 | `description` 이 사람용 주석이 아니라 지시다 |
| 2-1 추출 프롬프트 | 온톨로지+규칙 주입 | `build_extraction_prompt()` 를 모델에 이어 붙인다 |
| 2-1 판정 기준 | 어느 관계에도 안 맞는 진술을 어디에 적을지 정한다 | 대사·수송도 `BINDS` 로 |
| 2-2 CoT | 생각을 칸으로 쪼개기 | `reasoning` 칸을 `triples` 앞에 |
| 3-1 후처리 | 근거로 걸러내기 | 근거가 원문에 있는가, 그 근거가 이 트리플을 뒷받침하는가 |
| 3-1 시그니처 검사 | 방향까지 대조 | 관계 이름만 보면 절반만 보는 것 |
| 3-1 id 해소 | 이름을 id 로 | 못 붙으면 `:Candidate` 로 격리 |
| 3-1 배치 추출 | 문서 여럿을 한 번에 | `chain.batch([...])`, 결과는 입력 순서 그대로 |
| 3-1 동시 실행 제한 | 요청이 한꺼번에 나가지 않게 | `config={"max_concurrency": 2}` |
| 3-2 근거 등급 | 어디서 온 사실인가 | `curated` 는 확정, 논문은 낮추지 않는다 |
| 3-3 타입 계층 | 상위로 물으면 하위도 잡힘 | `:Gene:Enzyme` 겹친 레이블 |

- 온톨로지는 **코드 한 곳의 dict**: 여기만 고치면 프롬프트·적재가 함께 바뀝니다.
- 구조화 출력은 **칸의 모양**을 강제하지만 칸 사이의 약속은 못 봅니다. 그래서 시그니처 검사가 따로 필요합니다.
- **CoT 는 결과를 늘 좋게 만들지 않습니다.** 값어치는 `reasoning` 이 틀린 자리를 짚어 준다는 데 있습니다. 도움이 되는지는 문서·프롬프트·모델마다 달라 **재 봐야 압니다.**
- 붙일 곳을 못 찾은 사실도 **버리지 않고** 격리해 둡니다. 다음 단원들이 이어받습니다.
- 타입 계층으로 잃은 구분을 **일부** 되찾을 수 있지만 원래 자리만큼은 아닙니다.

## ⏭️ 예고: 다음 시간

오늘은 트리플을 **뽑아서 적재**하는 데까지 왔습니다. 그런데 뽑은 것이 얼마나 믿을 만한지는 아직 재지 않았습니다. 시그니처를 통과했다고 맞는 트리플인 것도 아니고, CoT 로 트리플이 줄었다고 좋아진 것도 아닙니다.

다음 시간에는 **관계 추출의 품질을 재는 법**을 배웁니다. 사람이 직접 만든 정답 목록(골드셋)을 오늘 쓴 바로 그 지문으로 만들고, 준수율·정밀도·재현율로 추출기를 채점합니다. 놓친 트리플이 없는지 확인하는 절차도 함께 봅니다.

수고하셨습니다!